# 01주 · 공학 모형·낙하산 운동

수치해석및실습 CE273 · 2027학년도 1학기

원본 슬라이드의 개념·예제·차시 순서에 연결한 **신규 병행 실습**. 원본 코드의 단순 복사본과 구분

**목표**: 수치 원리·도구 기능 이해 → 좋은 AI 질문·지시 → 오차·수렴·결과의 검토

**수업**: 2027-03-02 수치해석 소개 (1) · 대면 / 2027-03-04 공학 모형·수치해석 소개 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE273-numerical-analysis/week01/CE273_W01.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE273-numerical-analysis/week01)

슬라이드 예측 → 작은 손계산 → 실행 → 한 조건 변경 → AI 요청 수정 → 단위·오차·수렴·실패 상태 검토

셀 실행: Shift+Enter. 처음에는 위에서 아래로 실행. 공통 준비 셀 실행 후 각 절의 개별 실행도 가능. None 항목과 변경 셀은 직접 작성

## 0. 실행 환경·보조 함수

필요한 버전만 설치. 업데이트한 패키지를 이미 불러온 경우 런타임 재시작 후 처음부터 실행

보조 함수의 세부 문법 암기 불필요. 입력·처리·반환값·오차 기준·실패 조건부터 확인. 아래 실습의 호출 코드와 슬라이드 식 대조

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.11", "matplotlib": "3.8", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, linalg, interpolate, integrate

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 자료·최소 원소 수 확인 필요")
    return x

def relative_error(truth, estimate):
    # 기준값 0에서 상대오차를 정의하지 않음. 절대오차 별도 확인
    return np.nan if truth == 0 else abs((truth-estimate)/truth)
def parachute_velocity(t, g=9.81, m=68.1, c=12.5):
    if not (np.isfinite([g,m,c]).all() and g>0 and m>0 and c>=0):
        raise ValueError("g·m 양수, c 비음수 확인 필요")
    t = np.asarray(t, dtype=float)
    if not np.isfinite(t).all() or np.any(t<0):
        raise ValueError("유한한 비음수 시간 필요")
    return g*t if c == 0 else g*m/c*(-np.expm1(-c*t/m))

def parachute_distance(t, g=9.8, m=68.1, c=12.5):
    parachute_velocity(t,g,m,c) # 같은 물리 입력 검증
    t=np.asarray(t,dtype=float)
    if c == 0:
        return .5*g*t*t
    return g*m/c*(t + m/c*np.expm1(-c*t/m))
def ode_steps(f, start, y0, end, h, method="euler"):
    if not np.isfinite([start,y0,end,h]).all() or end<=start or h<=0:raise ValueError("유한한 초기조건·양의 구간·양의 h 필요")
    count=(end-start)/h;n=int(round(count))
    if not 1<=n<=100000 or not np.isclose(count,n,rtol=1e-12,atol=1e-12):raise ValueError("h로 구간을 정확히 분할할 양의 정수 스텝 수 필요")
    if method not in ["euler","heun","midpoint","ralston34","rk4"]:raise ValueError("방법 확인 필요")
    x=np.linspace(start,end,n+1);y=np.empty(n+1);y[0]=y0
    for i in range(n):
        k1=f(x[i],y[i])
        if method=="euler":slope=k1
        elif method=="heun":slope=(k1+f(x[i]+h,y[i]+h*k1))/2
        elif method=="midpoint":slope=f(x[i]+h/2,y[i]+h*k1/2)
        elif method=="ralston34":slope=k1/3+2*f(x[i]+3*h/4,y[i]+3*h*k1/4)/3
        else:
            k2=f(x[i]+h/2,y[i]+h*k1/2);k3=f(x[i]+h/2,y[i]+h*k2/2);k4=f(x[i]+h,y[i]+h*k3)
            slope=(k1+2*k2+2*k3+k4)/6
        y[i+1]=y[i]+h*slope
        if not np.isfinite(y[i+1]):raise ValueError("비유한 수치해: 스텝·모형·스케일 점검")
    return x,y


## S01. 모형·매개변수·단위와 해석해

**슬라이드 연결**: 2026_S1_NA_Wk1a_Intro1.pptx p19, 20 / 2026_S1_NA_Wk1b_Intro2.pptx p12, 13, 14, 15

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 초기속도·종단속도·시간 단위는?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
g,m,c=9.81,68.1,12.5
t=np.arange(0,11,2);exact=parachute_velocity(t,g,m,c)
rounded=53.44*(1-np.exp(-.18355*t))
print(pd.DataFrame({"t_s":t,"v_m_s":exact,"original_rounded":rounded,"difference":exact-rounded}))
print("종단속도(m/s):",g*m/c)
assert np.isclose(exact[0],0) and (exact<g*m/c).all()

**조건 변경**: g만 9.81→9.8로 변경. 알고리즘 차이와 입력 차이 구분

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: 선형 항력·m68.1kg·c12.5kg/s·g9.81m/s²·v(0)=0·시간s 명시

**결과 검토**: 원본 Excel의 53.44·0.18355는 반올림 상수. 새 정리본은 원래 매개변수로 계산

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## S02. Euler 갱신과 스텝 검토

**슬라이드 연결**: 2026_S1_NA_Wk1b_Intro2.pptx p16, 17, 18, 19

**자료 구분**: 원본 예제·수치·원리에 기반. 코드에 표시한 추가 검토는 조건 변경 실험

**먼저 예측**: 같은 모형에서 h를 줄이면 무엇이 바뀌는가?

단위·스텝·초기값·분모·조건 중 이번 예제의 핵심을 실행 전에 기록

In [ ]:
f=lambda t,v:9.81-12.5/68.1*v
for h in [2.,1.,.5]:
    t,v=ode_steps(f,0,0,10,h)
    error=abs(v[-1]-parachute_velocity(10))
    print({"h_s":h,"v10":v[-1],"absolute_error":error})
assert np.isclose(ode_steps(f,0,0,10,2)[1][1],19.62)
plt.plot(t,v,"o-",label="Euler");plt.plot(t,parachute_velocity(t),label="analytic");plt.xlabel("t (s)");plt.ylabel("v (m/s)");plt.legend();plt.show()

**조건 변경**: m·c는 고정하고 h만 변경. 실행 시간·오차 비교

원본 셀을 복사한 뒤 한 조건씩 변경. 기준 결과와 변경 결과를 구분

**AI 요청에 넣을 정보**: Euler 갱신식·초기값·스텝·종료시점·단위·해석해 대조 요청

**결과 검토**: 선형 항력 모형의 오차와 수치 근사의 오차 구분

예상·실행·독립 검산·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 바꾼 조건과 검산 코드를 이 셀에 추가


## 주차 확인

모형·단위·초기조건과 Euler 첫 갱신 설명

입력 항목:

- `초기속도_m_s`: 해당 계산값·항 수·오차 기준의 수치
- `Euler_첫스텝_h2_m_s`: 해당 계산값·항 수·오차 기준의 수치
- `스텝을_줄이면_모형오차도_자동으로_사라짐`: 판단 · Python `True`/`False`

공개 계산·개념 확인용 항목. 값만 복사하면 학습 완료로 판단 불가. 가정·적용 범위·검산 근거는 아래 서술과 함께 확인

In [ ]:
answers = {'초기속도_m_s': None, 'Euler_첫스텝_h2_m_s': None, '스텝을_줄이면_모형오차도_자동으로_사라짐': None}


In [ ]:
def check_answers(values):
    expected = {'초기속도_m_s': 0.0, 'Euler_첫스텝_h2_m_s': 19.62, '스텝을_줄이면_모형오차도_자동으로_사라짐': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

1. 문제·모형·입력·단위·자료 출처
2. 최초 요청: 목적·조건·알고리즘·출력·검증 기준
3. 실행 전 예측·작은 손계산의 근거
4. 의심한 출력·오차·수렴·실패 조건과 수정 지시
5. 채택·수정·거절한 내용, 독립 검산·한계
6. AI 없이 설명 가능한 원리 한 가지

글자 수 자동 채점 없음. 실행 성공·작은 갱신량·작은 잔차와 문제 해결의 타당성 구분

In [ ]:
review_log = {
    "문제·모형·자료·단위": None,
    "최초 AI 요청": None,
    "예측·손계산": None,
    "의심한 부분·수정 지시": None,
    "검산·최종 판단·한계": None,
    "내 설명": None,
}
